# CNN-LSTM Baseline — Kaggle Competition Submission

Notebook inference dành riêng cho **CNN + BiLSTM** đã train trong `04_student_cnn_lstm_kaggle.ipynb`.

Notebook này chạy trực tiếp trên Kaggle: đọc test DICOM từ `/kaggle/input`, dùng đúng preprocessing và model architecture của training notebook, rồi xuất `/kaggle/working/submission.csv`.

**Không cần tải DICOM về máy local.**

## Trước khi chạy

1. Add Input → Competition: `RSNA Knee Abnormality Detection`.
2. Add Input → Output của notebook `04_student_cnn_lstm_kaggle` (hoặc Dataset chứa `cnn_lstm_model_artifacts.zip`).
3. Bật GPU.
4. Không cần add pseudo-label Dataset trong notebook inference.

Khi kiểm tra interactive, Kaggle có thể chỉ cung cấp vài test studies mẫu. Khi dùng **Submit to Competition**, Kaggle rerun notebook trong môi trường chấm với hidden test set.

Notebook không cài package từ Internet. Điều này phù hợp với yêu cầu Code Competition chạy Internet disabled.

## 0. Runtime và imports

In [ ]:
import os
import gc
import re
import json
import math
import time
import shutil
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
from IPython.display import display, FileLink

import pydicom
from pydicom.pixels import apply_modality_lut
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

warnings.filterwarnings("ignore", category=UserWarning)

print("PyTorch:", torch.__version__)
print("pydicom:", pydicom.__version__)
print("OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Notebook Settings → Accelerator → GPU.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory / 1024**3:.1f} GiB")

## 1. Cấu hình, competition paths và model artifact

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
ID_COL = "StudyInstanceUID"
PLANE_ORDER = ["Sagittal", "Coronal", "Axial"]
PLANE_TO_ID = {"Sagittal": 0, "Coronal": 1, "Axial": 2}

IMG_SIZE = 224
SLICES_PER_SERIES = 12
MAX_SERIES_PER_STUDY = 3
MAX_TOTAL_SLICES = SLICES_PER_SERIES * MAX_SERIES_PER_STUDY
MIN_DECODED_SLICES_PER_SERIES = 4

BATCH_SIZE = 8
NUM_WORKERS = 2
PREPROCESS_SAVE_EVERY = 10

COMP_ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
if not (COMP_ROOT / "test.csv").exists():
    roots = list(Path("/kaggle/input/competitions").glob("*/test.csv"))
    if not roots:
        raise FileNotFoundError("Không thấy test.csv. Hãy Add Input competition RSNA Knee Abnormality Detection.")
    COMP_ROOT = roots[0].parent

TEST_CSV = COMP_ROOT / "test.csv"
TEST_SERIES_CSV = COMP_ROOT / "test_series.csv"
TEST_SERIES_DIR = COMP_ROOT / "test_series"
SAMPLE_SUBMISSION_CSV = COMP_ROOT / "sample_submission.csv"

OUTPUT_ROOT = Path("/kaggle/working/cnn_lstm_submission")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_ROOT / "test_preprocessed_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for required in [TEST_CSV, TEST_SERIES_CSV, TEST_SERIES_DIR, SAMPLE_SUBMISSION_CSV]:
    if not required.exists():
        raise FileNotFoundError(f"Missing competition input: {required}")

# Find model artifact without traversing the huge competition DICOM tree.
SEARCH_ROOTS = []
for name in ["notebooks", "datasets", "models"]:
    p = Path("/kaggle/input") / name
    if p.exists():
        SEARCH_ROOTS.append(p)
for p in Path("/kaggle/input").iterdir():
    if p.is_dir() and p.name not in {"competitions", "datasets", "notebooks", "models"}:
        SEARCH_ROOTS.append(p)

artifact_zips = []
checkpoint_files = []
for root in SEARCH_ROOTS:
    for current, dirs, files in os.walk(root):
        depth = len(Path(current).relative_to(root).parts)
        dirs[:] = [d for d in dirs if d not in {"train_series", "test_series", "preprocessed_cache"}]
        for f in files:
            full = Path(current) / f
            if f == "cnn_lstm_model_artifacts.zip":
                artifact_zips.append(full)
            elif f == "cnn_lstm_best.pt":
                checkpoint_files.append(full)
        if depth >= 5:
            dirs.clear()

MODEL_DIR = OUTPUT_ROOT / "model_artifact"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if checkpoint_files:
    BEST_CKPT = checkpoint_files[0]
elif artifact_zips:
    chosen_zip = artifact_zips[0]
    print("Found model ZIP:", chosen_zip)
    with zipfile.ZipFile(chosen_zip) as zf:
        members = [m for m in zf.namelist() if Path(m).name == "cnn_lstm_best.pt"]
        if not members:
            raise FileNotFoundError("cnn_lstm_model_artifacts.zip does not contain cnn_lstm_best.pt")
        with zf.open(members[0]) as src, open(MODEL_DIR / "cnn_lstm_best.pt", "wb") as dst:
            shutil.copyfileobj(src, dst)
    BEST_CKPT = MODEL_DIR / "cnn_lstm_best.pt"
else:
    print("Top-level Kaggle input folders:")
    for p in Path("/kaggle/input").iterdir():
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy cnn_lstm_best.pt hoặc cnn_lstm_model_artifacts.zip. "
        "Hãy Add Input từ Output của notebook CNN-LSTM đã Save Version."
    )

print("Competition root:", COMP_ROOT)
print("Test CSV:", TEST_CSV)
print("Test series CSV:", TEST_SERIES_CSV)
print("Model checkpoint:", BEST_CKPT)
print("Output root:", OUTPUT_ROOT)

## 2. Load test studies và kiểm tra submission schema

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV)

for df in [test_df, test_series_df, sample_submission]:
    if ID_COL in df.columns:
        df[ID_COL] = df[ID_COL].astype(str)

expected_columns = [ID_COL, *TARGETS]
if sample_submission.columns.tolist() != expected_columns:
    raise ValueError(
        "sample_submission columns do not match model target order.\n"
        f"Expected: {expected_columns}\nGot: {sample_submission.columns.tolist()}"
    )

if test_df[ID_COL].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID in test.csv")
if sample_submission[ID_COL].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID in sample_submission.csv")
if set(test_df[ID_COL]) != set(sample_submission[ID_COL]):
    raise ValueError(
        "UID mismatch between test.csv and sample_submission.csv: "
        f"test={len(test_df)}, sample={len(sample_submission)}"
    )

required_series_cols = {
    ID_COL, "SeriesInstanceUID", "Anatomical_Plane",
    "Fluid_Sensitive", "Fat_Suppression"
}
missing_series_cols = required_series_cols - set(test_series_df.columns)
if missing_series_cols:
    raise ValueError(f"test_series.csv missing columns: {sorted(missing_series_cols)}")

print("Test studies:", len(test_df))
print("Test series rows:", len(test_series_df))
print("Submission rows:", len(sample_submission))
print("Targets:", len(TARGETS))
print("UID/schema checks passed.")

if len(test_df) <= 3:
    print("NOTE: This appears to be Kaggle's visible sample test set. The hidden scoring run replaces it with the full test set.")

## 3. Chọn series cho mỗi study (giống training notebook)

In [ ]:
test_series_df["SeriesInstanceUID"] = test_series_df["SeriesInstanceUID"].astype(str)
test_series_df["plane_norm"] = test_series_df["Anatomical_Plane"].astype(str).str.strip().str.title()
test_series_df["fluid_num"] = pd.to_numeric(test_series_df["Fluid_Sensitive"], errors="coerce").fillna(-1)
test_series_df["fat_num"] = pd.to_numeric(test_series_df["Fat_Suppression"], errors="coerce").fillna(-1)

selected_rows = []
selected_by_uid = defaultdict(list)
plane_presence_rows = []

for uid, group in test_series_df.groupby(ID_COL, sort=False):
    present = []
    for plane in PLANE_ORDER:
        candidates = group[group["plane_norm"].eq(plane)].copy()
        if candidates.empty:
            continue
        candidates = candidates.sort_values(
            ["fluid_num", "fat_num", "SeriesInstanceUID"],
            ascending=[False, False, True],
            kind="mergesort",
        )
        chosen = candidates.iloc[0]
        item = {
            ID_COL: str(uid),
            "SeriesInstanceUID": str(chosen["SeriesInstanceUID"]),
            "Anatomical_Plane": plane,
            "plane_id": PLANE_TO_ID[plane],
            "Fluid_Sensitive": int(chosen["fluid_num"]) if chosen["fluid_num"] in (0, 1) else -1,
            "Fat_Suppression": int(chosen["fat_num"]) if chosen["fat_num"] in (0, 1) else -1,
        }
        selected_rows.append(item)
        selected_by_uid[str(uid)].append(item)
        present.append(plane)
    plane_presence_rows.append({ID_COL: str(uid), "n_planes_selected": len(present), "planes": ";".join(present)})

selected_test_series = pd.DataFrame(selected_rows)
plane_presence_df = pd.DataFrame(plane_presence_rows)
selected_test_series.to_csv(OUTPUT_ROOT / "selected_test_series.csv", index=False)

print("Selected series rows:", len(selected_test_series))
print("Studies with 3 planes:", int((plane_presence_df.n_planes_selected == 3).sum()))
print("Studies with 2 planes:", int((plane_presence_df.n_planes_selected == 2).sum()))
print("Studies with 1 plane:", int((plane_presence_df.n_planes_selected == 1).sum()))
print("Studies with 0 planes:", int((plane_presence_df.n_planes_selected == 0).sum()))
display(plane_presence_df.head())

## 4. DICOM decoding và preprocessing (copy cùng logic training)

In [ ]:
def _safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def read_sorted_dicom_headers(series_dir: Path):
    paths = sorted([
        p for p in series_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".dcm"
    ])
    if not paths:
        return []

    rows = []
    first_normal = None
    for path in paths:
        try:
            ds = pydicom.dcmread(
                str(path), stop_before_pixels=True, force=True,
                specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]
            )
            iop = getattr(ds, "ImageOrientationPatient", None)
            ipp = getattr(ds, "ImagePositionPatient", None)
            instance = _safe_float(getattr(ds, "InstanceNumber", np.nan))

            position_value = np.nan
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                row_cos = np.asarray(iop[:3], dtype=np.float64)
                col_cos = np.asarray(iop[3:6], dtype=np.float64)
                normal = np.cross(row_cos, col_cos)
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                    if first_normal is None:
                        first_normal = normal
                    position_value = float(np.dot(np.asarray(ipp[:3], dtype=np.float64), first_normal))

            rows.append({
                "path": path,
                "filename": path.name,
                "position": position_value,
                "instance": instance,
            })
        except Exception as exc:
            rows.append({
                "path": path, "filename": path.name,
                "position": np.nan, "instance": np.nan,
                "header_error": f"{type(exc).__name__}: {exc}",
            })

    valid_pos = sum(np.isfinite(r.get("position", np.nan)) for r in rows)
    valid_inst = sum(np.isfinite(r.get("instance", np.nan)) for r in rows)

    if valid_pos >= max(2, int(0.7 * len(rows))):
        rows.sort(key=lambda r: (
            np.inf if not np.isfinite(r.get("position", np.nan)) else r["position"],
            r["filename"]
        ))
        sort_mode = "ImagePositionPatient"
    elif valid_inst >= max(2, int(0.7 * len(rows))):
        rows.sort(key=lambda r: (
            np.inf if not np.isfinite(r.get("instance", np.nan)) else r["instance"],
            r["filename"]
        ))
        sort_mode = "InstanceNumber"
    else:
        def natural_key(name):
            return [int(part) if part.isdigit() else part.lower()
                    for part in re.split(r"(\d+)", name)]
        rows.sort(key=lambda r: natural_key(r["filename"]))
        sort_mode = "filename_fallback"

    for i, r in enumerate(rows):
        r["sort_index"] = i
        r["sort_mode"] = sort_mode
    return rows


def decode_dicom_slice(path: Path):
    ds = pydicom.dcmread(str(path), force=True)
    pixels = np.asarray(ds.pixel_array)

    if pixels.ndim == 3:
        if pixels.shape[-1] in (3, 4):
            pixels = pixels[..., :3].astype(np.float32)
            pixels = 0.299 * pixels[..., 0] + 0.587 * pixels[..., 1] + 0.114 * pixels[..., 2]
        else:
            pixels = pixels[pixels.shape[0] // 2]

    pixels = np.squeeze(pixels)
    if pixels.ndim != 2:
        raise ValueError(f"Expected 2D MRI slice, got shape={pixels.shape}")

    pixels = pixels.astype(np.float32)
    try:
        pixels = np.asarray(apply_modality_lut(pixels, ds), dtype=np.float32)
    except Exception:
        slope = _safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0)
        intercept = _safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0)
        pixels = pixels * slope + intercept

    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
        lo, hi = np.nanmin(pixels), np.nanmax(pixels)
        pixels = hi + lo - pixels

    pixels[~np.isfinite(pixels)] = 0
    return pixels


def normalize_resize_series(image_list, output_size=IMG_SIZE):
    if not image_list:
        return []
    all_values = np.concatenate([img.reshape(-1) for img in image_list])
    lo, hi = np.percentile(all_values, [1.0, 99.5])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(all_values)), float(np.max(all_values))
    if hi <= lo:
        hi = lo + 1.0

    output = []
    for img in image_list:
        x = np.clip(img, lo, hi)
        x = (x - lo) / (hi - lo)
        x = np.clip(x, 0, 1)
        x = (x * 255.0).round().astype(np.uint8)
        x = cv2.resize(x, (output_size, output_size), interpolation=cv2.INTER_AREA)
        output.append(x)
    return output


def process_one_test_series(series_row):
    uid = str(series_row[ID_COL])
    series_uid = str(series_row["SeriesInstanceUID"])
    plane = str(series_row["Anatomical_Plane"])
    series_dir = TEST_SERIES_DIR / uid / series_uid
    if not series_dir.exists():
        return [], [], f"missing_series_directory: {series_dir}"

    headers = read_sorted_dicom_headers(series_dir)
    if not headers:
        return [], [], "no_dicom_files"

    n = len(headers)
    target_n = min(SLICES_PER_SERIES, n)
    initial = np.unique(np.linspace(0, n - 1, target_n).round().astype(int)).tolist()
    initial_set = set(initial)
    candidates = initial + [i for i in range(n) if i not in initial_set]

    decoded, decoded_positions, errors = [], [], []
    for idx in candidates:
        try:
            img = decode_dicom_slice(headers[idx]["path"])
            decoded.append(img)
            decoded_positions.append(idx / max(n - 1, 1))
            if len(decoded) >= SLICES_PER_SERIES:
                break
        except Exception as exc:
            errors.append(f"{headers[idx]['filename']}: {type(exc).__name__}: {exc}")

    if len(decoded) < MIN_DECODED_SLICES_PER_SERIES:
        return [], [], "too_few_decoded_slices: " + " | ".join(errors[:2])

    order = np.argsort(decoded_positions)
    decoded = [decoded[i] for i in order]
    decoded_positions = [decoded_positions[i] for i in order]
    normalized = normalize_resize_series(decoded)

    plane_id = PLANE_TO_ID.get(plane, -1)
    fluid_value = _safe_float(series_row.get("Fluid_Sensitive", -1), -1)
    fat_value = _safe_float(series_row.get("Fat_Suppression", -1), -1)
    fluid = int(fluid_value) if fluid_value in (0, 1) else -1
    fat = int(fat_value) if fat_value in (0, 1) else -1

    metadata = [
        {"plane_id": plane_id, "fluid": fluid, "fat": fat, "position": float(pos)}
        for pos in decoded_positions
    ]
    return normalized, metadata, "ok"


def process_one_test_study(uid):
    image_list, metadata_list, used_series, errors = [], [], [], []
    selected = sorted(selected_by_uid.get(str(uid), []), key=lambda x: x["plane_id"])

    for item in selected:
        try:
            imgs, metas, status = process_one_test_series(pd.Series(item))
        except Exception as exc:
            imgs, metas = [], []
            status = f"exception:{type(exc).__name__}:{exc}"

        if status == "ok":
            image_list.extend(imgs)
            metadata_list.extend(metas)
            used_series.append(item["SeriesInstanceUID"])
        else:
            errors.append(f"{item['Anatomical_Plane']}:{item['SeriesInstanceUID']}:{status}")

    if not image_list:
        return None, {
            "status": "failed",
            "error": "; ".join(errors) or "no selected series",
            "n_slices": 0,
            "n_series_used": 0,
        }

    if len(image_list) > MAX_TOTAL_SLICES:
        image_list = image_list[:MAX_TOTAL_SLICES]
        metadata_list = metadata_list[:MAX_TOTAL_SLICES]

    images = np.zeros((MAX_TOTAL_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    plane_ids = np.full(MAX_TOTAL_SLICES, -1, dtype=np.int8)
    fluid_flags = np.full(MAX_TOTAL_SLICES, -1, dtype=np.int8)
    fat_flags = np.full(MAX_TOTAL_SLICES, -1, dtype=np.int8)
    positions = np.zeros(MAX_TOTAL_SLICES, dtype=np.float32)
    valid = np.zeros(MAX_TOTAL_SLICES, dtype=np.uint8)

    n_valid = len(image_list)
    images[:n_valid] = np.stack(image_list, axis=0)
    valid[:n_valid] = 1

    for i, meta in enumerate(metadata_list):
        plane_ids[i] = meta["plane_id"]
        fluid_flags[i] = meta["fluid"]
        fat_flags[i] = meta["fat"]
        positions[i] = meta["position"]

    return (images, plane_ids, fluid_flags, fat_flags, positions, valid), {
        "status": "ok",
        "error": "; ".join(errors),
        "n_slices": n_valid,
        "n_series_used": len(used_series),
        "used_series": ";".join(used_series),
    }

print("Test DICOM preprocessing functions are ready.")

## 5. Khởi tạo test cache và checkpoint

Test cache dùng cùng định dạng với training: `uint8`, tối đa 36 slices mỗi study, ảnh 224×224. Nếu session bị ngắt nhưng Kaggle Files persistence còn giữ output, các UID có `status=ok` sẽ được bỏ qua.

In [ ]:
N_TEST = len(test_df)
T = MAX_TOTAL_SLICES

volume_path = CACHE_DIR / "test_slices_uint8.npy"
valid_path = CACHE_DIR / "slice_valid.npy"
plane_path = CACHE_DIR / "plane_ids.npy"
fluid_path = CACHE_DIR / "fluid_flags.npy"
fat_path = CACHE_DIR / "fat_flags.npy"
position_path = CACHE_DIR / "slice_positions.npy"
status_path = CACHE_DIR / "test_preprocess_status.csv"

required_cache = [volume_path, valid_path, plane_path, fluid_path, fat_path, position_path]
cache_exists = all(p.exists() for p in required_cache)

if cache_exists:
    existing = np.load(volume_path, mmap_mode="r")
    if existing.shape != (N_TEST, T, IMG_SIZE, IMG_SIZE) or existing.dtype != np.uint8:
        print(
            f"Stale cache detected ({existing.shape}/{existing.dtype}) for N_TEST={N_TEST}; "
            "recreating cache for the current test set."
        )
        del existing
        for p in required_cache:
            if p.exists():
                p.unlink()
        if status_path.exists():
            status_path.unlink()
        cache_exists = False
    else:
        del existing

if cache_exists:
    volumes = np.lib.format.open_memmap(volume_path, mode="r+")
    valid_mm = np.lib.format.open_memmap(valid_path, mode="r+")
    plane_mm = np.lib.format.open_memmap(plane_path, mode="r+")
    fluid_mm = np.lib.format.open_memmap(fluid_path, mode="r+")
    fat_mm = np.lib.format.open_memmap(fat_path, mode="r+")
    position_mm = np.lib.format.open_memmap(position_path, mode="r+")
    print("Resuming existing test cache.")
else:
    required_gb = N_TEST * T * IMG_SIZE * IMG_SIZE / 1024**3
    free_gb = shutil.disk_usage(OUTPUT_ROOT).free / 1024**3
    print(f"Estimated test image cache: {required_gb:.2f} GiB | Free output disk: {free_gb:.2f} GiB")
    if free_gb < required_gb + 1.0:
        raise RuntimeError("Not enough output disk space for test cache.")

    volumes = np.lib.format.open_memmap(
        volume_path, mode="w+", dtype=np.uint8,
        shape=(N_TEST, T, IMG_SIZE, IMG_SIZE)
    )
    valid_mm = np.lib.format.open_memmap(valid_path, mode="w+", dtype=np.uint8, shape=(N_TEST, T))
    plane_mm = np.lib.format.open_memmap(plane_path, mode="w+", dtype=np.int8, shape=(N_TEST, T))
    fluid_mm = np.lib.format.open_memmap(fluid_path, mode="w+", dtype=np.int8, shape=(N_TEST, T))
    fat_mm = np.lib.format.open_memmap(fat_path, mode="w+", dtype=np.int8, shape=(N_TEST, T))
    position_mm = np.lib.format.open_memmap(position_path, mode="w+", dtype=np.float32, shape=(N_TEST, T))

    volumes[:] = 0
    valid_mm[:] = 0
    plane_mm[:] = -1
    fluid_mm[:] = -1
    fat_mm[:] = -1
    position_mm[:] = 0
    for arr in [volumes, valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
        arr.flush()

if status_path.exists():
    prior = pd.read_csv(status_path).fillna("")
    status_by_uid = {str(r[ID_COL]): r.to_dict() for _, r in prior.iterrows()}
else:
    status_by_uid = {}

print("Test studies:", N_TEST)
print("Completed test preprocessing records:", len(status_by_uid))
print("Cache directory:", CACHE_DIR)

## 6. Preprocess test DICOM — Batch 1

Xử lý studies từ vị trí 0 đến 499. Có checkpoint trong batch để có thể chạy lại cell nếu session bị ngắt.

In [ ]:
def run_test_preprocess_batch(start_idx, end_idx, batch_name):
    end_idx = min(end_idx, N_TEST)
    if start_idx >= end_idx:
        print(f"{batch_name}: no studies in this range ({start_idx}:{end_idx}).")
        return

    started = time.time()
    processed_now = 0
    uid_list = test_df[ID_COL].astype(str).tolist()

    print(f"{batch_name}: studies [{start_idx}, {end_idx}) / {N_TEST}")

    for row_idx in range(start_idx, end_idx):
        uid = uid_list[row_idx]
        if status_by_uid.get(uid, {}).get("status") == "ok":
            continue

        result, info = process_one_test_study(uid)

        if result is not None:
            images, plane_ids, fluid, fat, positions, valid = result
            volumes[row_idx] = images
            plane_mm[row_idx] = plane_ids
            fluid_mm[row_idx] = fluid
            fat_mm[row_idx] = fat
            position_mm[row_idx] = positions
            valid_mm[row_idx] = valid
        else:
            volumes[row_idx] = 0
            plane_mm[row_idx] = -1
            fluid_mm[row_idx] = -1
            fat_mm[row_idx] = -1
            position_mm[row_idx] = 0
            valid_mm[row_idx] = 0

        status_by_uid[uid] = {
            ID_COL: uid,
            "row_idx": row_idx,
            "status": info.get("status", "failed"),
            "n_slices": info.get("n_slices", 0),
            "n_series_used": info.get("n_series_used", 0),
            "error": info.get("error", ""),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        }

        processed_now += 1
        if processed_now % 5 == 0 or row_idx + 1 == end_idx:
            done = sum(v.get("status") == "ok" for v in status_by_uid.values())
            print(
                f"[{row_idx + 1}/{N_TEST}] overall_ok={done}/{N_TEST} "
                f"current_status={info.get('status')} "
                f"elapsed={(time.time()-started)/60:.1f} min",
                flush=True,
            )

        if processed_now % PREPROCESS_SAVE_EVERY == 0:
            for arr in [volumes, valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
                arr.flush()
            pd.DataFrame(list(status_by_uid.values())).to_csv(status_path, index=False)

    for arr in [volumes, valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
        arr.flush()
    pd.DataFrame(list(status_by_uid.values())).to_csv(status_path, index=False)
    print(f"{batch_name} completed. Newly processed: {processed_now}")

run_test_preprocess_batch(0, 500, "TEST PREPROCESS BATCH 1")

## 7. Preprocess test DICOM — Batch 2

In [ ]:
run_test_preprocess_batch(500, 1000, "TEST PREPROCESS BATCH 2")

## 8. Preprocess test DICOM — Batch 3 (remaining studies)

In [ ]:
run_test_preprocess_batch(1000, N_TEST, "TEST PREPROCESS BATCH 3")

## 9. Audit preprocessing

In [ ]:
status_df = pd.read_csv(status_path).fillna("")
status_df[ID_COL] = status_df[ID_COL].astype(str)
status_by_uid_final = set(status_df[ID_COL])
expected_test_ids = set(test_df[ID_COL].astype(str))

missing_status = expected_test_ids - status_by_uid_final
failed_preprocess = status_df[status_df["status"].ne("ok")].copy()

print("Expected test studies:", len(expected_test_ids))
print("Status records:", len(status_df))
print("Successful preprocessing:", int(status_df["status"].eq("ok").sum()))
print("Failed preprocessing:", len(failed_preprocess))
print("Missing status records:", len(missing_status))

if missing_status:
    raise RuntimeError(f"Missing preprocessing records for {len(missing_status)} test studies.")

if len(failed_preprocess):
    print("Failed studies will receive neutral 0.5 fallback predictions to keep the required submission schema.")
    display(failed_preprocess[[ID_COL, "n_slices", "error"]].head(20))

status_df.to_csv(OUTPUT_ROOT / "test_preprocess_status.csv", index=False)

## 10. Khôi phục đúng CNN + BiLSTM architecture và load checkpoint

Các class bên dưới được giữ đồng nhất với notebook training. Không huấn luyện lại trong notebook submission.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.bn2(self.conv2(x))
        return F.relu(x + identity, inplace=True)


class SliceCNN(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = nn.Sequential(ResidualBlock(32, 48, stride=1), ResidualBlock(48, 48))
        self.layer2 = nn.Sequential(ResidualBlock(48, 96, stride=2), ResidualBlock(96, 96))
        self.layer3 = nn.Sequential(ResidualBlock(96, 160, stride=2), ResidualBlock(160, 160))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(160, feature_dim)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        return self.proj(x)


class CNNBiLSTM(nn.Module):
    def __init__(self, num_targets=12, cnn_dim=128, plane_dim=8, hidden=128):
        super().__init__()
        self.cnn = SliceCNN(cnn_dim)
        self.plane_embedding = nn.Embedding(4, plane_dim, padding_idx=3)
        lstm_input = cnn_dim + plane_dim + 3
        self.lstm = nn.LSTM(
            input_size=lstm_input,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.20,
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(128, num_targets),
        )

    def forward(self, images, valid, plane_ids, fluid, fat, positions):
        B, T, H, W = images.shape
        valid_bool = valid.bool()
        flat_valid = valid_bool.reshape(-1)
        x = images.reshape(B * T, 1, H, W)[flat_valid].float().div_(255.0)
        x = F.interpolate(x, size=(160, 160), mode="bilinear", align_corners=False)
        encoded_valid = self.cnn(x)

        encoded = torch.zeros(
            (B * T, encoded_valid.shape[-1]),
            dtype=encoded_valid.dtype,
            device=encoded_valid.device,
        )
        encoded[flat_valid] = encoded_valid
        encoded = encoded.view(B, T, -1)

        pids = plane_ids.clamp(min=0, max=3)
        p_emb = self.plane_embedding(pids)
        meta = torch.stack([
            fluid.clamp(min=-1, max=1),
            fat.clamp(min=-1, max=1),
            positions.clamp(min=0, max=1),
        ], dim=-1)
        seq = torch.cat([encoded, p_emb, meta], dim=-1)

        lengths = valid_bool.sum(dim=1).clamp(min=1).cpu()
        packed = pack_padded_sequence(seq, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)

        time_mask = torch.arange(T, device=out.device).unsqueeze(0) < lengths.to(out.device).unsqueeze(1)
        scores = self.attention(out).squeeze(-1).masked_fill(~time_mask, -1e4)
        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(out * weights.unsqueeze(-1), dim=1)
        return self.classifier(pooled)


device = torch.device("cuda")
checkpoint = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)

checkpoint_targets = checkpoint.get("targets")
if checkpoint_targets != TARGETS:
    raise ValueError(
        "Checkpoint target order does not match sample submission. "
        f"Checkpoint={checkpoint_targets}; expected={TARGETS}"
    )

model = CNNBiLSTM(num_targets=len(TARGETS)).to(device)
model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()

print("CNN-BiLSTM checkpoint loaded.")
print("Best epoch:", checkpoint.get("best_epoch"))
print("Best validation BCE:", checkpoint.get("best_val_loss"))
print("Model device:", device)

## 11. Dataset và inference

In [ ]:
# Read-only memory maps for test cache.
volumes = np.load(volume_path, mmap_mode="r")
valid_cache = np.load(valid_path, mmap_mode="r")
plane_cache = np.load(plane_path, mmap_mode="r")
fluid_cache = np.load(fluid_path, mmap_mode="r")
fat_cache = np.load(fat_path, mmap_mode="r")
position_cache = np.load(position_path, mmap_mode="r")

uid_list = test_df[ID_COL].astype(str).tolist()
uid_to_idx = {uid: i for i, uid in enumerate(uid_list)}

class TestStudyDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        return {
            "images": torch.from_numpy(np.array(volumes[idx], copy=True)),
            "valid": torch.from_numpy(np.array(valid_cache[idx], copy=True).astype(np.uint8)),
            "plane_ids": torch.from_numpy(np.array(plane_cache[idx], copy=True).astype(np.int64)),
            "fluid": torch.from_numpy(np.array(fluid_cache[idx], copy=True).astype(np.float32)),
            "fat": torch.from_numpy(np.array(fat_cache[idx], copy=True).astype(np.float32)),
            "positions": torch.from_numpy(np.array(position_cache[idx], copy=True).astype(np.float32)),
            "uid": uid_list[idx],
        }


# Only pass studies with at least one successfully decoded slice through the network.
# Failed DICOM studies retain neutral 0.5 fallback predictions.
valid_indices = [i for i in range(N_TEST) if int(valid_cache[i].sum()) > 0]
failed_indices = sorted(set(range(N_TEST)) - set(valid_indices))

print("Studies with usable DICOM cache:", len(valid_indices))
print("Studies with no usable DICOM slices:", len(failed_indices))

pred_by_uid = {
    uid: np.full(len(TARGETS), 0.5, dtype=np.float32)
    for uid in uid_list
}

inference_dataset = TestStudyDataset(valid_indices)
inference_loader = DataLoader(
    inference_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

model.eval()
started = time.time()

with torch.inference_mode():
    for step, batch in enumerate(inference_loader, start=1):
        images = batch["images"].to(device, non_blocking=True)
        valid = batch["valid"].to(device, non_blocking=True)
        plane_ids = batch["plane_ids"].to(device, non_blocking=True)
        fluid = batch["fluid"].to(device, non_blocking=True)
        fat = batch["fat"].to(device, non_blocking=True)
        positions = batch["positions"].to(device, non_blocking=True)

        logits = model(images, valid, plane_ids, fluid, fat, positions)
        probs = torch.sigmoid(logits.float()).cpu().numpy()

        for uid, prob in zip(batch["uid"], probs):
            pred_by_uid[str(uid)] = prob.astype(np.float32)

        if step % 25 == 0 or step == len(inference_loader):
            print(
                f"Inference batch {step}/{len(inference_loader)} "
                f"| studies={min(step*BATCH_SIZE, len(valid_indices))}/{len(valid_indices)} "
                f"| elapsed={(time.time()-started)/60:.1f} min",
                flush=True,
            )

print("Inference finished in", round((time.time() - started) / 60, 2), "minutes")

## 12. Tạo `submission.csv` theo đúng template

In [ ]:
# Preserve the exact sample_submission UID order and column order.
submission = sample_submission[[ID_COL]].copy()

probability_frame = pd.DataFrame.from_dict(
    pred_by_uid,
    orient="index",
    columns=TARGETS,
)
probability_frame.index.name = ID_COL
probability_frame = probability_frame.reset_index()
probability_frame[ID_COL] = probability_frame[ID_COL].astype(str)

submission = submission.merge(
    probability_frame,
    on=ID_COL,
    how="left",
    validate="one_to_one",
)
submission = submission[expected_columns]

# Validate before writing.
if submission[expected_columns].isna().any().any():
    missing = submission.loc[submission[TARGETS].isna().any(axis=1), ID_COL].tolist()
    raise ValueError(f"Missing predictions for UIDs: {missing[:10]}")

values = submission[TARGETS].to_numpy(dtype=np.float64)
if not np.isfinite(values).all():
    raise ValueError("Submission contains NaN/Inf values")
if (values < 0).any() or (values > 1).any():
    raise ValueError("Submission probabilities must lie in [0, 1]")
if submission[ID_COL].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID in submission")
if submission[ID_COL].tolist() != sample_submission[ID_COL].tolist():
    raise ValueError("Submission UID order differs from sample_submission")

SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.8f")

print("Created:", SUBMISSION_PATH)
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("Prediction range:", float(values.min()), "to", float(values.max()))
print("Fallback 0.5 studies:", len(failed_indices))
display(submission.head())

## 13. Export inference log và package phụ trợ

In [ ]:
submission_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "architecture": "small_residual_CNN + BiLSTM + attention pooling",
    "checkpoint": str(BEST_CKPT),
    "checkpoint_best_epoch": checkpoint.get("best_epoch"),
    "checkpoint_validation_bce": checkpoint.get("best_val_loss"),
    "competition_root": str(COMP_ROOT),
    "n_test_studies": int(N_TEST),
    "n_preprocessed_ok": int(len(valid_indices)),
    "n_preprocess_failed_fallback_0_5": int(len(failed_indices)),
    "targets": TARGETS,
    "preprocessing": {
        "img_size": IMG_SIZE,
        "slices_per_series": SLICES_PER_SERIES,
        "max_series_per_study": MAX_SERIES_PER_STUDY,
        "selected_planes": PLANE_ORDER,
        "series_selection": "highest Fluid_Sensitive then Fat_Suppression, one series per plane",
    },
    "submission_metric": "macro-average ROC-AUC across 12 targets",
}

manifest_path = OUTPUT_ROOT / "submission_manifest.json"
manifest_path.write_text(
    json.dumps(submission_manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

predictions_path = OUTPUT_ROOT / "test_predictions_probabilities.csv"
submission.to_csv(predictions_path, index=False, float_format="%.8f")

failed_path = OUTPUT_ROOT / "test_preprocess_failures.csv"
status_df[status_df["status"].ne("ok")].to_csv(failed_path, index=False)

bundle_path = OUTPUT_ROOT / "cnn_lstm_submission_artifacts.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [SUBMISSION_PATH, predictions_path, status_path, failed_path, manifest_path,
                 OUTPUT_ROOT / "selected_test_series.csv"]:
        if path.exists():
            zf.write(path, arcname=path.name)

print("Submission file:", SUBMISSION_PATH)
print("Auxiliary package:", bundle_path)
display(FileLink(SUBMISSION_PATH))
display(FileLink(bundle_path))

## 14. Submit lên competition

1. Kiểm tra `submission.csv` xuất hiện trong `/kaggle/working/`.
2. Chọn **Save Version** → **Save & Run All (Commit)**.
3. Trong Notebook, chọn **Submit to Competition** (hoặc nút Submit tương ứng trên trang competition).
4. Chờ Kaggle rerun notebook trong môi trường scoring.
5. Mở Competition → My Submissions/Leaderboard để xem public score.

### Hai lưu ý quan trọng

- Khi chạy interactive, `test.csv` có thể chỉ gồm 3 study ví dụ. Điểm số có ý nghĩa chỉ xuất hiện sau khi Kaggle rerun với hidden test set.
- Competition đánh giá bằng macro-average ROC-AUC. Ta xuất xác suất sigmoid 0–1, không threshold thành nhãn 0/1.

Notebook phải chạy dưới giới hạn 9 giờ và Internet disabled. Không thêm `pip install`/download model từ Internet vào submission run. Model checkpoint phải được attach như Kaggle Input.